# NV-Generate-CTMR en Google Colab (12 GB de VRAM)

Este cuaderno prepara una **generación de CT sintética 3D** con `rflow-ct`, a partir del repositorio ya clonado. El modelo también dispone de variantes para generar MRI, pero **no realiza una traducción CT↔MR** ni convierte una CT de entrada en una MRI emparejada.

> **Límite de memoria:** el README del proyecto recomienda al menos 16 GB para el quick start. El modo `--low-vram` traslada modelos entre CPU y GPU y puede ayudar, pero **12 GB no es una configuración garantizada ni un preset validado**. Empezaremos con la salida CT más pequeña permitida por el modelo (`256×256×128`). Si ocurre OOM, usa una GPU Colab con más VRAM; bajar más la resolución no es una solución válida para esta variante CT.

Ejecuta todas las celdas únicamente en Colab. No se instala ni se ejecuta nada en tu computadora local. Uso educativo e investigación; no usar para diagnóstico ni decisiones clínicas. Revisa las licencias del código, los pesos y las dependencias antes de descargarlos.

## 1. Configurar el entorno de Google Colab con GPU

En Colab, selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**. Ejecuta este diagnóstico antes de instalar dependencias o descargar pesos. La memoria libre cambia durante la sesión; la comprobación no predice por sí sola si la inferencia cabrá.

In [ ]:
import subprocess
import sys

print("Python:", sys.version)
subprocess.run(["nvidia-smi"], check=False)

try:
    import torch
except ImportError as exc:
    raise RuntimeError(
        "No se encuentra PyTorch en este runtime. Confirma que Colab tiene GPU habilitada."
    ) from exc

print("PyTorch:", torch.__version__)
print("CUDA compilada en PyTorch:", torch.version.cuda)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    for device_index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(device_index)
        free_bytes, total_bytes = torch.cuda.mem_get_info(device_index)
        print(
            f"GPU {device_index}: {props.name}; "
            f"VRAM total={total_bytes / 2**30:.1f} GiB; "
            f"libre ahora={free_bytes / 2**30:.1f} GiB"
        )
else:
    print("Activa una GPU en Colab antes de continuar con el modelo.")

Monta Drive y ajusta `REPO_DIR` a la carpeta donde ya está clonado el repositorio. Colab **no puede acceder a rutas locales de tu computadora**: si el clon solo está en este equipo, súbelo/cópialo a Drive o clónalo dentro de la sesión de Colab. Si el clon solo está en el disco temporal de otra sesión, tampoco estará disponible aquí.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

# Cambia esta ruta si el repositorio está en otra carpeta de Drive.
REPO_DIR = Path("/content/drive/MyDrive/NV-Generate-CTMR")
required_files = [
    "requirements.txt",
    "scripts/diff_model_infer.py",
    "scripts/download_model_data.py",
    "configs/config_network_rflow.json",
    "configs/config_maisi_diff_model_rflow-ct.json",
    "configs/environment_maisi_diff_model_rflow-ct.json",
]
missing_files = [str(REPO_DIR / name) for name in required_files if not (REPO_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        "No encuentro el clon o faltan archivos requeridos. Ajusta REPO_DIR.\n"
        + "\n".join(missing_files)
    )

os.chdir(REPO_DIR)
print("Repositorio:", Path.cwd())
print("Archivos clave verificados:", len(required_files))

## 3. Instalar dependencias solo en la sesión de Colab

La instalación ocurre en la máquina virtual temporal de Colab. No fijamos ni reemplazamos PyTorch a ciegas: Colab ya provee una compilación CUDA, y los requisitos del proyecto solo exigen `torch>=2.1`. Si la instalación avisa de conflictos en Colab, no actualices `torch` manualmente; revisa la compatibilidad antes de continuar.

In [ ]:
%pip install -r requirements.txt

import matplotlib
import nibabel
import monai
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("MONAI:", monai.__version__)
print("nibabel:", nibabel.__version__)
print("matplotlib:", matplotlib.__version__)

if not torch.cuda.is_available():
    raise RuntimeError("Este notebook requiere un runtime de Colab con GPU NVIDIA/CUDA.")

## 4. Verificar estructura y definir rutas de trabajo

Los pesos se descargan una vez a `models/` dentro del clon, usando el mecanismo del repositorio. La salida final se escribirá en Drive para que sobreviva al cierre de Colab; archivos temporales y configuración editada vivirán en `/content`.

In [ ]:
import os
from pathlib import Path

WORK_DIR = Path("/content/nv_generate_ctmr_run")
OUTPUT_DIR = Path("/content/drive/MyDrive/NV-Generate-CTMR-results")
WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ["MONAI_DATA_DIRECTORY"] = str(WORK_DIR / "monai_data")
Path(os.environ["MONAI_DATA_DIRECTORY"]).mkdir(parents=True, exist_ok=True)

NETWORK_CONFIG = REPO_DIR / "configs/config_network_rflow.json"
SOURCE_INFER_CONFIG = REPO_DIR / "configs/config_maisi_diff_model_rflow-ct.json"
SOURCE_ENV_CONFIG = REPO_DIR / "configs/environment_maisi_diff_model_rflow-ct.json"

for config_path in [NETWORK_CONFIG, SOURCE_INFER_CONFIG, SOURCE_ENV_CONFIG]:
    if not config_path.is_file():
        raise FileNotFoundError(f"Falta configuración requerida: {config_path}")

print("Directorio temporal:", WORK_DIR)
print("Salida persistente:", OUTPUT_DIR)
print("Los pesos se descargarán al directorio models/ del repositorio.")

## 5. Preparar checkpoints y parámetros de inferencia

Este flujo usa el checkpoint preentrenado de `rflow-ct` y no entrena ni actualiza pesos. Para bajar el uso de memoria, configuramos una única muestra, 30 pasos (valor requerido por RFlow) y `dim=[256,256,128]`, el tamaño mínimo permitido para CT en esta variante. El espaciado conserva un FOV corporal razonable: `dim × spacing = 384 × 384 × 384 mm`.

In [ ]:
import json

INFER_CONFIG = WORK_DIR / "config_maisi_diff_model_rflow-ct.json"
ENV_CONFIG = WORK_DIR / "environment_maisi_diff_model_rflow-ct.json"

with SOURCE_INFER_CONFIG.open(encoding="utf-8") as file:
    infer_config = json.load(file)
inference_params = infer_config.get("diffusion_unet_inference")
if not isinstance(inference_params, dict):
    raise KeyError("No encuentro el bloque diffusion_unet_inference en la configuración upstream.")

required_params = {"dim", "spacing", "modality", "random_seed", "num_inference_steps"}
missing_params = required_params.difference(inference_params)
if missing_params:
    raise KeyError(f"Faltan parámetros esperados en la configuración: {sorted(missing_params)}")

inference_params.update({
    "dim": [256, 256, 128],
    "spacing": [1.5, 1.5, 3.0],
    "modality": 1,
    "random_seed": 0,
    "num_inference_steps": 30,
    "cfg_guidance_scale": 0,
})
if "num_output_samples" in inference_params:
    inference_params["num_output_samples"] = 1
elif "num_output_samples" in infer_config:
    infer_config["num_output_samples"] = 1

with INFER_CONFIG.open("w", encoding="utf-8") as file:
    json.dump(infer_config, file, indent=2)

with SOURCE_ENV_CONFIG.open(encoding="utf-8") as file:
    env_config = json.load(file)
env_config["output_dir"] = str(OUTPUT_DIR)
with ENV_CONFIG.open("w", encoding="utf-8") as file:
    json.dump(env_config, file, indent=2)

print("Inferencia:", inference_params)
print("FOV (mm):", [
    dim * spacing for dim, spacing in zip(inference_params["dim"], inference_params["spacing"])
])
print("Configuraciones runtime:", INFER_CONFIG, ENV_CONFIG)

Antes de descargar, confirma que tienes almacenamiento suficiente en Drive y que aceptaste las condiciones de uso de los pesos de NVIDIA. Esta descarga se ejecuta en Colab y puede tardar; `--model_only` evita bajar las bases de datos auxiliares que no necesita esta inferencia.

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "scripts.download_model_data",
        "--version",
        "rflow-ct",
        "--root_dir",
        "./",
        "--model_only",
    ],
    cwd=REPO_DIR,
    check=True,
)

checkpoint_paths = [
    REPO_DIR / env_config["existing_ckpt_filepath"],
    REPO_DIR / env_config["trained_autoencoder_path"],
]
missing_checkpoints = [path for path in checkpoint_paths if not path.is_file()]
if missing_checkpoints:
    raise FileNotFoundError(
        "La descarga terminó, pero no encuentro estos checkpoints:\n"
        + "\n".join(str(path) for path in missing_checkpoints)
    )
print("Checkpoints listos:")
for checkpoint_path in checkpoint_paths:
    print("-", checkpoint_path)

## 6. Ajustar el uso de VRAM para una GPU de 12 GB

Para esta ruta, las medidas compatibles con el código upstream son: `--low-vram` y una dimensión pequeña. La inferencia image-only no expone en este CLI ajustes de batch, autocast ni ventanas del VAE; el script gestiona inferencia sin gradientes. No se fuerza precisión mixta manualmente, porque podría alterar compatibilidad o calidad.

La dimensión CT mínima válida es `256×256×128`; bajar de ella viola las restricciones del modelo. `torch.cuda.empty_cache()` solo libera bloques no usados por el allocator y no reduce la memoria que requiere el modelo. Si la GPU se queda sin memoria, no hay una reducción adicional de resolución que podamos recomendar como válida: reinicia Colab y/o cambia a una GPU de 16 GB o más.

In [ ]:
import torch

torch.cuda.empty_cache()
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f"VRAM libre antes de descarga/inferencia: {free_bytes / 2**30:.2f} GiB")
print(f"VRAM total: {total_bytes / 2**30:.2f} GiB")
print("Modo de ejecución previsto: --low-vram; una muestra; dim=[256, 256, 128].")
print("Esta comprobación no garantiza que el modelo quepa en la GPU.")

## 7. Ejecutar inferencia CT desde el notebook

Este comando usa el script oficial de inferencia image-only para generar una CT sintética, no una CT↔MR. La primera ejecución carga los checkpoints y puede tardar varios minutos. Los logs se muestran en la celda; si aparece CUDA OOM, consulta la sección 9.

In [ ]:
import time

inference_command = [
    sys.executable,
    "-m",
    "scripts.diff_model_infer",
    "-t",
    "./configs/config_network_rflow.json",
    "-e",
    str(ENV_CONFIG),
    "-c",
    str(INFER_CONFIG),
    "--version",
    "rflow-ct",
    "--low-vram",
]
print("Comando:", " ".join(inference_command))
start_time = time.perf_counter()
inference_result = subprocess.run(inference_command, cwd=REPO_DIR, check=False)
print(f"Tiempo total: {(time.perf_counter() - start_time) / 60:.1f} minutos")
if inference_result.returncode != 0:
    raise RuntimeError(
        f"La inferencia terminó con código {inference_result.returncode}. "
        "Revisa el log anterior y la sección 9 antes de reintentar."
    )
print("Archivos de salida:", OUTPUT_DIR)

In [ ]:
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np

ct_files = sorted(OUTPUT_DIR.glob("*.nii.gz"), key=lambda path: path.stat().st_mtime)
if not ct_files:
    raise FileNotFoundError(f"No encuentro volúmenes NIfTI en {OUTPUT_DIR}")

ct_path = ct_files[-1]
ct_image = nib.load(str(ct_path))
ct_data = ct_image.dataobj
mid_x, mid_y, mid_z = [size // 2 for size in ct_image.shape[:3]]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for axis, image_slice, title in zip(
    axes,
    [np.asarray(ct_data[mid_x, :, :]), np.asarray(ct_data[:, mid_y, :]), np.asarray(ct_data[:, :, mid_z])],
    [f"Sagital x={mid_x}", f"Coronal y={mid_y}", f"Axial z={mid_z}"],
):
    axis.imshow(image_slice.T, cmap="gray", origin="lower", vmin=-1000, vmax=1000)
    axis.set_title(title)
    axis.axis("off")

fig.suptitle(f"CT sintética: {ct_path.name}")
plt.tight_layout()
plt.show()
print("NIfTI:", ct_path)
print("Shape:", ct_image.shape)
print("Espaciado (mm):", ct_image.header.get_zooms()[:3])

## 8. Guardar resultados y metadatos en Google Drive

Los volúmenes ya se generan directamente en `OUTPUT_DIR` de Drive. Esta celda copia las configuraciones exactas usadas y guarda un manifiesto con variante, semilla, resolución, FOV y GPU para poder reproducir el experimento.

In [ ]:
import shutil
from datetime import datetime, timezone

shutil.copy2(INFER_CONFIG, OUTPUT_DIR / "config_maisi_diff_model_rflow-ct.json")
shutil.copy2(ENV_CONFIG, OUTPUT_DIR / "environment_maisi_diff_model_rflow-ct.json")

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "repository": "https://github.com/NVIDIA-Medtech/NV-Generate-CTMR",
    "variant": "rflow-ct",
    "inference_mode": "image-only; synthetic CT generation (not CT-to-MR translation)",
    "dim": inference_params["dim"],
    "spacing_mm": inference_params["spacing"],
    "field_of_view_mm": [
        dim * spacing for dim, spacing in zip(inference_params["dim"], inference_params["spacing"])
    ],
    "seed": inference_params["random_seed"],
    "num_inference_steps": inference_params["num_inference_steps"],
    "low_vram": True,
    "gpu": torch.cuda.get_device_name(0),
    "nifti_outputs": [path.name for path in OUTPUT_DIR.glob("*.nii.gz")],
}
with (OUTPUT_DIR / "run_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2, ensure_ascii=False)

print("Artefactos persistidos en:", OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(f"- {path.name} ({path.stat().st_size / 2**20:.1f} MiB)")

## 9. Diagnóstico de memoria, CUDA y rutas

- **CUDA out of memory:** no repitas la inferencia automáticamente. Reinicia el runtime de Colab, vuelve a seleccionar GPU, comprueba la VRAM libre y ejecuta una sola vez. `--low-vram` y `dim=[256,256,128]` ya están aplicados. Si persiste, necesitas una GPU de Colab con más VRAM; bajar más `dim` no es válido para CT.
- **CUDA no disponible o incompatibilidad de versiones:** confirma que el runtime es GPU y revisa `torch.__version__` y `torch.version.cuda`. No reinstales PyTorch a ciegas; si el runtime quedó inconsistente, restablécelo y vuelve a correr las celdas desde el principio.
- **Ruta o archivo inexistente:** revisa `REPO_DIR`; el clon debe contener `scripts/` y `configs/`. Si el repo está en Drive, confirma que Drive esté montado. Los checkpoints descargados solo persisten si el clon está en Drive.
- **Falta de espacio:** revisa el almacenamiento disponible en Drive y en `/content`; los pesos ocupan varios GB.

Para un reintento controlado tras OOM: reinicia el runtime, vuelve a correr las celdas en orden hasta la inferencia y verifica que se use `--low-vram`. No cambies la configuración de resolución a valores fuera de las restricciones del modelo.

Documentación upstream: [repositorio](https://github.com/NVIDIA-Medtech/NV-Generate-CTMR), [guía de inferencia](https://github.com/NVIDIA-Medtech/NV-Generate-CTMR/blob/main/docs/inference.md), [rendimiento y memoria](https://github.com/NVIDIA-Medtech/NV-Generate-CTMR/blob/main/docs/performance.md), [modelo NV-Generate-CT](https://huggingface.co/nvidia/NV-Generate-CT), [licencia del código](https://github.com/NVIDIA-Medtech/NV-Generate-CTMR/blob/main/LICENSE) y [licencia de los pesos](https://github.com/NVIDIA-Medtech/NV-Generate-CTMR/blob/main/LICENSE.weights).